# Hybrid AI Floor Plan Analysis Pipeline

This notebook runs the floor plan analysis pipeline on Kaggle.

**Features:**
- YOLOv8-Seg room detection
- OCR text extraction (PaddleOCR/EasyOCR)
- Dimension parsing (feet/inches)
- Frontend-ready JSON output

## Setup Instructions

1. Upload the `pipeline` folder as a Kaggle dataset
2. Upload your YOLOv8 model (.pt file) as a dataset
3. Upload test images/PDFs as a dataset
4. Update the paths in Cell 2 below

In [ ]:
# Cell 1: Install additional dependencies
!pip install -q paddleocr paddlepaddle-gpu easyocr pymupdf shapely
print("Dependencies installed!")

In [ ]:
# Cell 2: Configuration - UPDATE THESE PATHS!
import os
import sys

# =============================================================================
# UPDATE THESE PATHS TO MATCH YOUR KAGGLE DATASETS
# =============================================================================

# Path to your uploaded pipeline code
PIPELINE_PATH = "/kaggle/input/sap-floor-plan-pipeline/pipeline"

# Path to your YOLOv8 segmentation model
MODEL_PATH = "/kaggle/input/sap-floor-plan-pipeline/pipeline/results/yolov8s-seg.pt"

# Path to test images folder
IMAGES_PATH = "/kaggle/input/sap-floor-plan-pipeline/pipeline/Dataset/AAHHP"

# Path to test PDFs folder (optional)
PDF_PATH = "/kaggle/input/sap-floor-plan-pipeline/pipeline/Dataset/PDF"

# Output directory (Kaggle working directory)
OUTPUT_DIR = "/kaggle/working/output"

# =============================================================================
# Setup environment
# =============================================================================

# Set Ultralytics config directory to avoid permission issues
os.environ["YOLO_CONFIG_DIR"] = "/kaggle/working/Ultralytics"
os.environ["ULTRALYTICS_CONFIG_DIR"] = "/kaggle/working/Ultralytics"

# Add pipeline to Python path
sys.path.insert(0, PIPELINE_PATH)

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs("/kaggle/working/Ultralytics", exist_ok=True)

print(f"Pipeline path: {PIPELINE_PATH}")
print(f"Model path: {MODEL_PATH}")
print(f"Images path: {IMAGES_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print("\nSetup complete!")

In [ ]:
# Cell 3: Verify paths and list available files
import os
from pathlib import Path

def list_files(path, pattern="*", limit=10):
    """List files in directory."""
    p = Path(path)
    if not p.exists():
        print(f"Path not found: {path}")
        return []
    files = list(p.glob(pattern))[:limit]
    for f in files:
        print(f"  {f.name}")
    return files

print("=" * 50)
print("Available Images:")
print("=" * 50)
images = list_files(IMAGES_PATH, "*.png") + list_files(IMAGES_PATH, "*.PNG")

print("\n" + "=" * 50)
print("Available PDFs:")
print("=" * 50)
pdfs = list_files(PDF_PATH, "*.pdf")

print("\n" + "=" * 50)
print("Model file:")
print("=" * 50)
if Path(MODEL_PATH).exists():
    print(f"  {Path(MODEL_PATH).name} ({Path(MODEL_PATH).stat().st_size / 1024 / 1024:.1f} MB)")
else:
    print(f"  Model not found at {MODEL_PATH}")

In [ ]:
# Cell 4: Run pipeline on a SINGLE IMAGE
import subprocess

# Select an image to process
TEST_IMAGE = f"{IMAGES_PATH}/E-01.png"  # Change this to your image

cmd = [
    "python", f"{PIPELINE_PATH}/scripts/run_pipeline.py",
    "--model", MODEL_PATH,
    "--image", TEST_IMAGE,
    "--output-dir", OUTPUT_DIR,
    "--frontend-format",
    "--include-debug",
    "--device", "cuda",
    "--use-gpu",
]

print(f"Processing: {TEST_IMAGE}")
print("Command:", " ".join(cmd))
print("\n" + "=" * 50)

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Errors/Warnings:")
    print(result.stderr[-2000:])  # Last 2000 chars of stderr

In [ ]:
# Cell 5: Display visualization result
from IPython.display import Image, display
from pathlib import Path

# Find the result image
result_images = list(Path(OUTPUT_DIR).glob("*_result.png"))

if result_images:
    print(f"Displaying: {result_images[0].name}")
    display(Image(str(result_images[0]), width=800))
else:
    print("No result images found. Check the pipeline output above for errors.")

In [ ]:
# Cell 6: Load and display JSON results
import json
from pathlib import Path

# Find frontend JSON output
json_files = list(Path(OUTPUT_DIR).glob("*_frontend.json"))

if json_files:
    with open(json_files[0], "r") as f:
        result = json.load(f)
    
    page = result["pages"][0]
    summary = page["summary"]
    
    print("=" * 60)
    print("ANALYSIS SUMMARY")
    print("=" * 60)
    print(f"Schema Version: {result['schema_version']}")
    print(f"Source File: {result['source']['file']}")
    print(f"Total Rooms: {summary['total_rooms']}")
    print(f"Rooms with Labels: {summary['rooms_with_labels']}")
    print(f"Rooms with Dimensions: {summary['rooms_with_dimensions']}")
    print(f"Text Strategy: {summary['text_strategy']}")
    
    if summary.get("warnings"):
        print(f"\nWarnings: {summary['warnings']}")
    
    print("\n" + "=" * 60)
    print("ROOM DETAILS")
    print("=" * 60)
    
    for room in page["rooms"]:
        label = room.get("label", "Unknown")
        dims = room.get("dimensions", "N/A")
        conf = room["confidence"]["geometry"]
        area_px = room["geometry"]["area_pixels"]
        
        print(f"\nRoom {room['id']}: {label}")
        print(f"  Dimensions: {dims}")
        print(f"  Confidence: {conf:.1%}")
        print(f"  Area (px): {area_px:,.0f}")
        
        if room.get("dimensions_parsed"):
            dp = room["dimensions_parsed"]
            print(f"  Parsed: {dp['width_ft']}'{dp['width_in']}\" x {dp['height_ft']}'{dp['height_in']}\"")
            print(f"  Area (sqft): {dp['area_sqft']:.1f}")
else:
    print("No frontend JSON found. Checking for standard output...")
    json_files = list(Path(OUTPUT_DIR).glob("*_data.json"))
    if json_files:
        print(f"Found: {json_files[0]}")

In [ ]:
# Cell 7: Process MULTIPLE IMAGES (batch mode)
import subprocess

BATCH_OUTPUT = "/kaggle/working/batch_output"
os.makedirs(BATCH_OUTPUT, exist_ok=True)

cmd = [
    "python", f"{PIPELINE_PATH}/scripts/run_pipeline.py",
    "--model", MODEL_PATH,
    "--dataset-dir", IMAGES_PATH,
    "--output-dir", BATCH_OUTPUT,
    "--frontend-format",
    "--device", "cuda",
    "--use-gpu",
]

print(f"Processing all images in: {IMAGES_PATH}")
print("This may take a few minutes...\n")

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("Errors:")
    print(result.stderr[-2000:])

In [ ]:
# Cell 8: Process a PDF (Full Pipeline)
import subprocess

PDF_OUTPUT = "/kaggle/working/pdf_output"
os.makedirs(PDF_OUTPUT, exist_ok=True)

# Select a PDF to process
TEST_PDF = f"{PDF_PATH}/Test_5.pdf"  # Change this to your PDF

cmd = [
    "python", f"{PIPELINE_PATH}/scripts/run_pdf_pipeline.py",
    "--model", MODEL_PATH,
    "--pdf", TEST_PDF,
    "--output-dir", PDF_OUTPUT,
    "--device", "cuda",
    "--use-gpu",
]

print(f"Processing PDF: {TEST_PDF}")
print("This may take several minutes for multi-page PDFs...\n")

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("Errors:")
    print(result.stderr[-2000:])

In [ ]:
# Cell 9: List all output files
from pathlib import Path
import os

def show_outputs(directory):
    """Display all output files in directory."""
    p = Path(directory)
    if not p.exists():
        print(f"Directory not found: {directory}")
        return
    
    files = sorted(p.rglob("*"))
    files = [f for f in files if f.is_file()]
    
    print(f"\nFiles in {directory}:")
    print("-" * 60)
    
    for f in files[:30]:  # Limit to 30 files
        rel_path = f.relative_to(p)
        size_kb = f.stat().st_size / 1024
        print(f"  {rel_path} ({size_kb:.1f} KB)")
    
    if len(files) > 30:
        print(f"  ... and {len(files) - 30} more files")

print("=" * 60)
print("OUTPUT FILES")
print("=" * 60)

show_outputs(OUTPUT_DIR)
show_outputs(BATCH_OUTPUT) if os.path.exists(BATCH_OUTPUT) else None
show_outputs(PDF_OUTPUT) if os.path.exists(PDF_OUTPUT) else None

In [ ]:
# Cell 10: Download results (creates a zip file)
import shutil
from pathlib import Path

# Create zip of all outputs
zip_path = "/kaggle/working/floor_plan_results"

# Collect all output directories
output_dirs = [OUTPUT_DIR]
if Path(BATCH_OUTPUT).exists():
    output_dirs.append(BATCH_OUTPUT)
if Path(PDF_OUTPUT).exists():
    output_dirs.append(PDF_OUTPUT)

# Copy to single folder and zip
combined_dir = Path("/kaggle/working/combined_results")
combined_dir.mkdir(exist_ok=True)

for src_dir in output_dirs:
    if Path(src_dir).exists():
        dst = combined_dir / Path(src_dir).name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src_dir, dst)

shutil.make_archive(zip_path, 'zip', combined_dir)

print(f"Results packaged: {zip_path}.zip")
print(f"Size: {Path(zip_path + '.zip').stat().st_size / 1024 / 1024:.2f} MB")
print("\nDownload from the 'Output' tab on the right panel.")

## Output Format

The pipeline produces frontend-ready JSON with this structure:

```json
{
  "schema_version": "1.0.0",
  "source": {"type": "image", "file": "E-01.png"},
  "pages": [{
    "summary": {
      "total_rooms": 8,
      "rooms_with_labels": 8,
      "rooms_with_dimensions": 3
    },
    "rooms": [{
      "id": 1,
      "label": "Bed Room",
      "dimensions": "12'-0\" x 10'-6\"",
      "dimensions_parsed": {
        "width_total_inches": 144,
        "height_total_inches": 126,
        "area_sqft": 126.0
      },
      "confidence": {"geometry": 0.95, "label": 0.90},
      "geometry": {"polygon": [[x,y],...], "bbox": {...}}
    }]
  }]
}
```